##GPT-2 From Scratch

In [ ]:
import numpy as np
import math

np.random.seed(42)

# Model hyperparameters
d_model     = 32      # D  - embedding / residual stream width
num_heads   = 4        # H  - attention heads
d_head      = d_model // num_heads   # Dh = D / H = 8
hidden_dim  = 128       # Hd - feedforward hidden width
num_layers  = 4        # number of transformer blocks
max_seq_len = 64       # longest position table we allocate
block_size  = 32       # T  - training context length
batch_size  = 16       # B

# AdamW hyperparameters
learning_rate = 6e-4
beta1, beta2  = 0.9, 0.999
eps_adam      = 1e-8
weight_decay  = 0.01

num_epochs = 2000

In [ ]:
# Tokenizer

text = """to be, or not to be, that is the question,
whether tis nobler in the mind to suffer
the slings and arrows of outrageous fortune,
or to take arms against a sea of troubles
and by opposing end them. to die, to sleep,
no more, and by a sleep to say we end
the heart ache and the thousand natural shocks
that flesh is heir to. tis a consummation
devoutly to be wished. to die, to sleep,
to sleep, perchance to dream, ay theres the rub"""

chars = sorted(list(set(text)))
vocab_size = len(chars)                    # V, set from actual data
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    """string -> list[int] token ids"""
    return [stoi[c] for c in s]

def decode(ids):
    """list[int] -> string"""
    return "".join(itos[i] for i in ids)

token_ids = np.array(encode(text), dtype=np.int64)   # shape: (num_tokens,)
print("vocab_size:", vocab_size)
print("num_tokens:", len(token_ids))
print("sample encode:", encode("to be"))

vocab_size: 27
num_tokens: 430
sample encode: [22, 17, 1, 5, 8]


In [ ]:
class TokenEmbedding:
  def __init__(self, vocab_size, d_model):
    self.W = np.random.randn(vocab_size, d_model) * 0.02
    self.dW = np.zeros_like(self.W)

  def forward(self, token_ids):
    self.cache = token_ids
    return self.W[token_ids]

  def backward(self, dout):
    self.dW.fill(0)
    np.add.at(self.dW, self.cache, dout)
    return self.dW

In [ ]:
class PositionEmbedding:
  def __init__(self, max_seq_len, d_model):
      self.W = np.random.randn(max_seq_len, d_model) * 0.02
      self.dW = np.zeros_like(self.W)

  def forward(self, seq_len):
      self.seq_len = seq_len
      return self.W[:seq_len]

  def backward(self, dout):
    self.dW.fill(0)
    self.dW[:self.seq_len] = dout.sum(axis=0)
    return self.dW

In [ ]:
class InputEmbedding:
  def __init__(self, vocab_size, d_model, max_seq_len):
    self.token_emb = TokenEmbedding(vocab_size, d_model)
    self.pos_emb = PositionEmbedding(max_seq_len, d_model)

  def forward(self, token_ids):
    B, T = token_ids.shape
    tok = self.token_emb.forward(token_ids)
    pos = self.pos_emb.forward(T)
    return tok + pos[None, :, :]

  def backward(self, dout):
    self.token_emb.backward(dout)
    self.pos_emb.backward(dout)

In [ ]:
class LayerNorm:
  def __init__(self, d_model, eps=1e-5):
    self.gamma = np.ones(d_model)
    self.beta = np.zeros(d_model)
    self.eps = eps

  def forward(self, x):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    x_hat = (x-mu) / np.sqrt(var + self.eps)
    out = self.gamma * x_hat + self.beta
    self.cache = (x_hat, var, x)
    return out

  def backward(self, dout):
    x_hat, var, x = self.cache
    D = x.shape[-1]
    reduce_axes = tuple(range(dout.ndim - 1))

    self.dgamma = np.sum(dout * x_hat, axis=reduce_axes)
    self.dbeta = np.sum(dout, axis = reduce_axes)

    dx_hat = dout * self.gamma
    std_inv = 1.0 / np.sqrt(var + self.eps)
    dx = (1.0/D) * std_inv * (D * dx_hat - np.sum(dx_hat, axis=-1, keepdims=True)
    - x_hat * np.sum(dx_hat * x_hat, axis=-1, keepdims=True))
    return dx

In [ ]:
class SelfAttention:
  def forward(self, Q, K, V):
    d_head = Q.shape[-1]
    T = Q.shape[1]
    scores = Q @ K.transpose(0,2,1) / math.sqrt(d_head)
    mask = np.triu(np.ones((T,T)), k=1).astype(bool)
    scores = np.where(mask[None, :, :], -1e9, scores)
    scores_max = scores.max(axis=-1, keepdims=True)
    exp_scores = np.exp(scores - scores_max)
    attn = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
    out = attn @ V
    self.cache = (Q,K,V,attn,mask)
    return out

  def backward(self, dout):
    Q,K,V,attn,mask = self.cache
    d_head = Q.shape[-1]
    dV = attn.transpose(0,2,1) @ dout
    dattn = dout @ V.transpose(0,2,1)
    dscores = attn * (dattn - np.sum(dattn * attn, axis=-1, keepdims=True))
    dscores = np.where(mask[None, :, :], 0, dscores)
    dscores = dscores / math.sqrt(d_head)
    dQ = dscores @ K
    dK = dscores.transpose(0,2,1) @ Q
    return dQ, dK, dV

In [ ]:
class MultiHeadAttention:
  def __init__(self, d_model, num_heads):
    self.d_model = d_model
    self.num_heads = num_heads
    self.d_head = d_model // num_heads
    scale = 0.02
    self.W_Q = np.random.randn(d_model, d_model) * scale
    self.W_K = np.random.randn(d_model, d_model) * scale
    self.W_V = np.random.randn(d_model, d_model) * scale
    self.W_O = np.random.randn(d_model, d_model) * scale
    self.heads = [SelfAttention() for _ in range(num_heads)]

  def split_heads(self, x):
    B, T, _ = x.shape
    x = x.reshape(B, T, self.num_heads, self.d_head)
    return x.transpose(2,0,1,3)   #(H, B, T, d_head)

  def combine_heads(self, x):
    x = x.transpose(1,2,0,3)
    B, T = x.shape[0], x.shape[1]
    return x.reshape(B, T, self.d_model)

  def forward(self, x):
    self.x = x
    Q = x @ self.W_Q; K = x @ self.W_K; V = x @ self.W_V
    Qh, Kh, Vh = self.split_heads(Q), self.split_heads(K), self.split_heads(V)
    outs = [self.heads[i].forward(Qh[i], Kh[i], Vh[i]) for i in range(self.num_heads)]
    concat = self.combine_heads(np.stack(outs, axis=0))
    self.concat = concat
    return concat @ self.W_O

  def backward(self, dout):
    dconcat = dout @ self.W_O.T
    self.dW_O = self.concat.reshape(-1, self.d_model).T @ dout.reshape(-1, self.d_model)
    dheads = self.split_heads(dconcat)
    dQh, dKh, dVh = [], [], []
    for i in range(self.num_heads):
      dQ_i, dK_i, dV_i = self.heads[i].backward(dheads[i])
      dQh.append(dQ_i); dKh.append(dK_i); dVh.append(dV_i)
    dQ = self.combine_heads(np.stack(dQh, axis=0))
    dK = self.combine_heads(np.stack(dKh, axis=0))
    dV = self.combine_heads(np.stack(dVh, axis=0))
    x_flat = self.x.reshape(-1, self.d_model)
    self.dW_Q = x_flat.T @ dQ.reshape(-1, self.d_model)
    self.dW_K = x_flat.T @ dK.reshape(-1, self.d_model)
    self.dW_V = x_flat.T @ dV.reshape(-1, self.d_model)
    return dQ @ self.W_Q.T + dK @ self.W_K.T + dV @ self.W_V.T

In [ ]:
class FeedForward:
  def __init__(self, d_model, hidden_dim):
    scale = 0.02
    self.W1 = np.random.randn(d_model, hidden_dim) * scale
    self.b1 = np.zeros(hidden_dim)
    self.W2 = np.random.randn(hidden_dim, d_model) * scale
    self.b2 = np.zeros(d_model)

  def forward(self, x):
    self.x = x
    self.z1 = x @ self.W1 + self.b1
    self.a1 = np.maximum(0, self.z1)
    return self.a1 @ self.W2 + self.b2

  def backward(self, dout):
    D = self.x.shape[-1]
    Hd = self.a1.shape[-1]
    a1_flat = self.a1.reshape(-1, Hd)
    dout_flat = dout.reshape(-1, D)
    self.dW2 = a1_flat.T @ dout_flat
    self.db2 = dout_flat.sum(axis=0)

    da1 = dout @ self.W2.T
    dz1 = da1 * (self.z1 > 0)

    x_flat = self.x.reshape(-1, D)
    dz1_flat = dz1.reshape(-1, Hd)
    self.dW1 = x_flat.T @ dz1_flat
    self.db1 = dz1_flat.sum(axis=0)
    return dz1 @ self.W1.T

In [ ]:
class TransformerBlock:
  def __init__(self, d_model, num_heads, hidden_dim):
    self.ln1 = LayerNorm(d_model)
    self.attn = MultiHeadAttention(d_model, num_heads)
    self.ln2 = LayerNorm(d_model)
    self.ffn = FeedForward(d_model, hidden_dim)

  def forward(self, x):
    normed1 = self.ln1.forward(x)
    attn_out = self.attn.forward(normed1)
    x2 = x + attn_out   #residual 1
    self.x2 = x2

    normed2 = self.ln2.forward(x2)
    ffn_out = self.ffn.forward(normed2)
    return x2 + ffn_out     #residual 2

  def backward(self, dout):
    # residual 2: out = x2 + ffn_out
    dnormed2 = self.ffn.backward(dout)
    dx2_via_ffn = self.ln2.backward(dnormed2)
    dx2 = dout + dx2_via_ffn      # sum both paths

    #residual 1: x2= x + attn_out
    dnormed1 = self.attn.backward(dx2)
    dx_via_attn = self.ln1.backward(dnormed1)
    dx = dx2 + dx_via_attn        # sum both paths
    return dx

In [ ]:
class GPT2:
    def __init__(self, vocab_size, d_model, num_heads, hidden_dim, num_layers, max_seq_len):
        self.embed = InputEmbedding(vocab_size, d_model, max_seq_len)
        self.blocks = [TransformerBlock(d_model, num_heads, hidden_dim) for _ in range(num_layers)]
        self.ln_f = LayerNorm(d_model)
        self.b_lm = np.zeros(vocab_size)

    def forward(self, token_ids):
        x = self.embed.forward(token_ids)
        for block in self.blocks:
            x = block.forward(x)
        self.final_x = x
        self.normed = self.ln_f.forward(x)

        W_lm = self.embed.token_emb.W.T
        logits = self.normed @ W_lm + self.b_lm
        return logits

    def backward(self, dlogits):
        D = self.normed.shape[-1]
        normed_flat = self.normed.reshape(-1, D)
        dlogits_flat = dlogits.reshape(-1, dlogits.shape[-1])

        dW_lm = normed_flat.T @ dlogits_flat
        self.db_lm = dlogits_flat.sum(axis=0)

        W_lm = self.embed.token_emb.W.T
        dnormed = dlogits @ W_lm.T        # fixed: 3D, not flattened

        dx = self.ln_f.backward(dnormed)
        for block in reversed(self.blocks):
            dx = block.backward(dx)

        self.embed.backward(dx)
        self.embed.token_emb.dW += dW_lm.T

In [ ]:
def softmax(logits):
  m = logits.max(axis=-1, keepdims=True)
  e = np.exp(logits - m)
  return e / e.sum(axis=-1, keepdims=True)

def cross_entropy_loss(logits, target):
  B, T, V = logits.shape
  probs = softmax(logits)
  N = B * T

  b_idx = np.arange(B)[:, None]
  t_idx = np.arange(T)[None, :]
  correct_probs = probs[b_idx, t_idx, target]
  loss = -np.log(correct_probs + 1e-9).mean()

  dlogits = probs.copy()
  dlogits[b_idx, t_idx, target] -= 1
  dlogits /= N
  return loss, dlogits

In [ ]:
class AdamW:
  def __init__(self, lr=6e-4, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
    self.lr = lr
    self.beta1, self.beta2 = beta1, beta2
    self.eps = eps
    self.weight_decay = weight_decay
    self.m = {}
    self.v = {}
    self.t = 0

  def step(self, params):
    self.t += 1
    for key, p, g in params:
      if key not in self.m:
        self.m[key] = np.zeros_like(p)
        self.v[key] = np.zeros_like(p)

      self.m[key] = self.beta1*self.m[key] + (1-self.beta1)*g
      self.v[key] = self.beta2*self.v[key] + (1-self.beta2)*(g**2)

      m_hat = self.m[key] / (1-self.beta1**self.t)
      v_hat = self.v[key] / (1-self.beta2**self.t)

      p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
      p -= self.lr * self.weight_decay * p

In [ ]:
def get_params(model):
      params = []
      params.append(("tok_emb", model.embed.token_emb.W, model.embed.token_emb.dW))
      params.append(("pos_emb", model.embed.pos_emb.W, model.embed.pos_emb.dW))
      for i, block in enumerate(model.blocks):
          params.append((f"ln1_gamma{i}", block.ln1.gamma, block.ln1.dgamma))
          params.append((f"ln1_beta{i}", block.ln1.beta, block.ln1.dbeta))
          params.append((f"W_Q{i}", block.attn.W_Q, block.attn.dW_Q))
          params.append((f"W_K{i}", block.attn.W_K, block.attn.dW_K))
          params.append((f"W_V{i}", block.attn.W_V, block.attn.dW_V))
          params.append((f"W_O{i}", block.attn.W_O, block.attn.dW_O))
          params.append((f"ln2_gamma{i}", block.ln2.gamma, block.ln2.dgamma))
          params.append((f"ln2_beta{i}", block.ln2.beta, block.ln2.dbeta))
          params.append((f"ffn_W1_{i}", block.ffn.W1, block.ffn.dW1))
          params.append((f"ffn_b1_{i}", block.ffn.b1, block.ffn.db1))
          params.append((f"ffn_W2_{i}", block.ffn.W2, block.ffn.dW2))
          params.append((f"ffn_b2_{i}", block.ffn.b2, block.ffn.db2))
      params.append(("lnf_gamma", model.ln_f.gamma, model.ln_f.dgamma))
      params.append(("lnf_beta", model.ln_f.beta, model.ln_f.dbeta))
      params.append(("b_lm", model.b_lm, model.db_lm))
      return params

In [ ]:
class TextDataset:
  def __init__(self, token_ids, block_size):
    self.token_ids = token_ids
    self.block_size = block_size

  def get_batch(self, batch_size):
    max_start = len(self.token_ids) - self.block_size
    starts = np.random.randint(0, max_start, size=batch_size)
    x = np.stack([self.token_ids[s : s+self.block_size] for s in starts])
    y = np.stack([self.token_ids[s+1 : s+1+self.block_size] for s in starts])
    return x, y

In [ ]:
model = GPT2(vocab_size, d_model, num_heads, hidden_dim, num_layers, max_seq_len)
optimizer = AdamW(lr=learning_rate, beta1=beta1, beta2=beta2, eps=eps_adam, weight_decay=weight_decay)
dataset = TextDataset(token_ids, block_size)

for epoch in range(num_epochs):
  xb, yb = dataset.get_batch(batch_size)

  logits = model.forward(xb)
  loss, dlogits = cross_entropy_loss(logits, yb)

  model.backward(dlogits)
  optimizer.step(get_params(model))

  if epoch % 100 == 0:
    print(f"epoch {epoch} | loss {loss: .4f}")

epoch 0 | loss  3.2865
epoch 100 | loss  2.3756
epoch 200 | loss  1.6327
epoch 300 | loss  1.0632
epoch 400 | loss  0.6761
epoch 500 | loss  0.5168
epoch 600 | loss  0.3390
epoch 700 | loss  0.2687
epoch 800 | loss  0.2374
epoch 900 | loss  0.2231
epoch 1000 | loss  0.1750
epoch 1100 | loss  0.1635
epoch 1200 | loss  0.1726
epoch 1300 | loss  0.1722
epoch 1400 | loss  0.1766
epoch 1500 | loss  0.1545
epoch 1600 | loss  0.1378
epoch 1700 | loss  0.1644
epoch 1800 | loss  0.1290
epoch 1900 | loss  0.1338


In [ ]:
def generate(model, prompt, max_new_tokens=40, temperature=1.0, greedy=True):
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        context = np.array(ids[-max_seq_len:])[None, :]
        logits = model.forward(context)
        last_logits = logits[0, -1, :] / temperature

        probs = softmax(last_logits[None, :])[0]
        if greedy:
            next_id = int(np.argmax(probs))
        else:
            next_id = int(np.random.choice(len(probs), p=probs))
        ids.append(next_id)
    return decode(ids)

In [ ]:
prompt = "the"
output = generate(
    model,
    prompt,
    max_new_tokens=50,
    temperature=0.8,
    greedy=False
)

print(output)

the is the question,
whether tis tis a noble r  minob
